# Retail Demand Forecasting - Exploratory Data Analysis

This notebook performs comprehensive EDA on the UCI Online Retail II dataset.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
from src.data.ingestion import DataIngestion
from src.data.cleaning import DataCleaner
from src.data.validation import DataValidation
from src.data.aggregation import ForecastingDatasetBuilder
from src.visualization.plots import Visualizer

plt.style.use('seaborn-v0_8')
%matplotlib inline
FIG_DIR = Path('..') / 'reports' / 'figures'
viz = Visualizer(save_path=str(FIG_DIR))
cleaner = DataCleaner()
validator = DataValidation()
agg_builder = ForecastingDatasetBuilder()
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 1. Data Loading

In [ ]:
ingestion = DataIngestion('../data/raw')
data = ingestion.load_data('online_retail')
df = data['transactions']
print(f'Loaded {len(df):,} transactions')
print(f'Columns: {list(df.columns)}')

## 2. Data Overview

In [ ]:
print(f'Shape: {df.shape}')
print(f'\nFirst 5 rows:')
display(df.head())
print(f'\nData types:')
print(df.dtypes)

## 3. Data Quality Check

In [ ]:
print(validator.generate_validation_report(df, 'Online Retail II'))

In [ ]:
issues = validator.validate_online_retail(df)
print('Dataset issues found:')
for k, v in issues.items():
    print(f'  {k}: {v}')

## 4. Data Cleaning

In [ ]:
df_clean = cleaner.clean_online_retail(df)
print(f'After cleaning: {len(df_clean):,} rows (from {len(df):,})')
print(f'Unique SKUs: {df_clean["stockcode"].nunique():,}')
print(f'Unique Customers: {df_clean["customer_id"].nunique():,}')
print(f'Unique Countries: {df_clean["country"].nunique():,}')

## 5. Date Range and Transaction Volume

In [ ]:
dates = pd.to_datetime(df_clean['invoicedate'])
print(f'Date range: {dates.min()} to {dates.max()}')
print(f'Unique days: {dates.dt.date.nunique()}')

daily_trans = df_clean.groupby(dates.dt.date).size()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily_trans.index, daily_trans.values, linewidth=0.8)
ax.set_title('Daily Transaction Volume')
ax.set_xlabel('Date')
ax.set_ylabel('Number of Transactions')
ax.grid(True, alpha=0.3)
plt.tight_layout()
viz.save_figure(fig, 'daily_transactions.png')
plt.show()

## 6. Aggregate to Daily SKU Demand

In [ ]:
daily = agg_builder.build_daily_sku_demand(df_clean)
print(f'Aggregated dataset: {len(daily):,} rows')
print(f'Unique SKUs: {daily["stockcode"].nunique():,}')
print(f'Unique days: {daily["date"].nunique()}')
display(daily.head(10))

## 7. Daily Sales Trend

In [ ]:
daily_sum = daily.groupby('date')['daily_demand'].sum().reset_index()
daily_sum.columns = ['date', 'sales']
fig = viz.plot_time_series(daily_sum, 'date', 'sales')
viz.save_figure(fig, 'daily_sales_trend.png')
plt.show()

## 8. Revenue Trend

In [ ]:
revenue_daily = daily.groupby('date')['revenue'].sum().reset_index()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(revenue_daily['date'], revenue_daily['revenue'], color='green', linewidth=0.8)
ax.set_title('Daily Revenue')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue')
ax.grid(True, alpha=0.3)
plt.tight_layout()
viz.save_figure(fig, 'revenue_trend.png')
plt.show()

## 9. Seasonality Analysis

In [ ]:
fig = viz.plot_weekly_seasonality(daily_sum, 'date', 'sales')
viz.save_figure(fig, 'weekly_seasonality.png')
plt.show()

In [ ]:
fig = viz.plot_monthly_seasonality(daily_sum, 'date', 'sales')
viz.save_figure(fig, 'monthly_seasonality.png')
plt.show()

## 10. Top-Selling SKUs

In [ ]:
top_skus = daily.groupby('stockcode')['daily_demand'].sum().sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(range(len(top_skus)), top_skus.values, color='teal', alpha=0.7)
ax.set_xticks(range(len(top_skus)))
ax.set_xticklabels(top_skus.index, rotation=45, ha='right')
ax.set_xlabel('StockCode')
ax.set_ylabel('Total Demand')
ax.set_title('Top 20 SKUs by Total Demand')
ax.grid(True, alpha=0.3)
plt.tight_layout()
viz.save_figure(fig, 'top_skus.png')
plt.show()

## 11. Country Distribution

In [ ]:
country_qty = df_clean.groupby('country')['quantity'].sum().sort_values(ascending=False).head(20)
fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(range(len(country_qty)), country_qty.values, color='purple', alpha=0.7)
ax.set_xticks(range(len(country_qty)))
ax.set_xticklabels(country_qty.index, rotation=45, ha='right')
ax.set_xlabel('Country')
ax.set_ylabel('Total Quantity')
ax.set_title('Top 20 Countries by Sales Quantity')
ax.grid(True, alpha=0.3)
plt.tight_layout()
viz.save_figure(fig, 'country_distribution.png')
plt.show()

## 12. Sales Distribution

In [ ]:
fig = viz.plot_sales_distribution(daily_sum, 'sales')
viz.save_figure(fig, 'sales_distribution.png')
plt.show()

## 13. Rolling Averages

In [ ]:
daily_sum['rolling_7'] = daily_sum['sales'].rolling(7).mean()
daily_sum['rolling_30'] = daily_sum['sales'].rolling(30).mean()
daily_sum['rolling_90'] = daily_sum['sales'].rolling(90).mean()
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(daily_sum['date'], daily_sum['sales'], alpha=0.4, linewidth=0.5, label='Daily')
ax.plot(daily_sum['date'], daily_sum['rolling_7'], label='7-day MA', linewidth=1.5)
ax.plot(daily_sum['date'], daily_sum['rolling_30'], label='30-day MA', linewidth=2)
ax.plot(daily_sum['date'], daily_sum['rolling_90'], label='90-day MA', linewidth=2)
ax.set_title('Rolling Averages - Daily Demand')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
viz.save_figure(fig, 'rolling_averages.png')
plt.show()

## 14. Correlation Analysis

In [ ]:
daily_numeric = daily.select_dtypes(include=[np.number])
if len(daily_numeric.columns) > 1:
    fig = viz.plot_heatmap(daily_numeric, 'Feature Correlation Heatmap')
    viz.save_figure(fig, 'correlation_heatmap.png')
    plt.show()

## 15. Price Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df_clean['price'].clip(0, df_clean['price'].quantile(0.95)), bins=50, edgecolor='black', alpha=0.7)
ax.set_title('Price Distribution (95th percentile capped)')
ax.set_xlabel('Price')
ax.set_ylabel('Frequency')
ax.grid(True, alpha=0.3)
plt.tight_layout()
viz.save_figure(fig, 'price_distribution.png')
plt.show()

## 16. SKU Count per Day

In [ ]:
sku_per_day = daily.groupby('date')['stockcode'].nunique()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(sku_per_day.index, sku_per_day.values, color='orange', linewidth=0.8)
ax.set_title('Number of Active SKUs per Day')
ax.set_xlabel('Date')
ax.set_ylabel('Active SKUs')
ax.grid(True, alpha=0.3)
plt.tight_layout()
viz.save_figure(fig, 'sku_per_day.png')
plt.show()

## 17. Summary

In [ ]:
print('=== EDA Summary ===')
print(f'Raw transactions: {len(df):,}')
print(f'After cleaning: {len(df_clean):,}')
print(f'Daily SKU records: {len(daily):,}')
print(f'Unique SKUs: {daily["stockcode"].nunique():,}')
print(f'Date range: {daily["date"].min()} to {daily["date"].max()}')
print(f'Avg daily demand: {daily_sum["sales"].mean():.2f}')
print(f'Top SKU: {top_skus.index[0]} ({top_skus.values[0]:,.0f} total)')
print(f'Top country: {country_qty.index[0]} ({country_qty.values[0]:,.0f} qty)')
print(f'\nFigures saved to: {FIG_DIR.resolve()}')